# Wykład 1: Wprowadzenie do analizy danych tabelarycznych

## 1.1 Środowisko pracy

W trakcie zajęć będziemy pracować w środowisku **notatników obliczeniowych** Jupyter Notebook. Jest to forma dokumentu, która w jednym miejscu integruje:

* tekst (opis problemu, komentarze, wzory matematyczne),
* kod w języku Python,
* wyniki obliczeń (tabele, wykresy, wizualizacje).

Takie podejście jest standardem we współczesnej analizie danych, ponieważ jest wygodne, przejrzyste oraz umożliwia reprodukowalność analiz.

Na potrzeby zajęć korzystamy z platformy Google Colab, która:

* działa bezpośrednio w przeglądarce internetowej,
* nie wymaga lokalnej instalacji Pythona ani bibliotek,
* posiada wstępnie zainstalowane najważniejsze pakiety używane w analizie danych.

### Biblioteki i instrukcja `import`

Python w podstawowej wersji oferuje ogólne narzędzia programistyczne. Do specjalistycznych zadań — takich jak analiza danych — korzystamy z **bibliotek** (zwanych też modułami lub pakietami): gotowych zbiorów kodu, które udostępniają nowe funkcje i struktury danych. Bibliotekę ładujemy instrukcją `import`, która sprawia, że jej zawartość staje się dostępna w bieżącym notatniku.

Instrukcja `import` ma kilka wariantów:

```python
import pandas            # cała biblioteka, dostęp przez pandas.read_csv()
import pandas as pd      # cała biblioteka pod krótszym aliasem: pd.read_csv()
from scipy import stats  # tylko wybrany moduł z większej biblioteki
```

W tym kursie najczęściej będziemy korzystać z drugiej formy — importu z aliasem. Aliasy takie jak `pd` dla pandas czy `np` dla NumPy są powszechnie przyjętymi konwencjami, które spotkamy w dokumentacji i w cudzym kodzie.

Nasza pierwsza komórka z kodem:

In [1]:
import pandas as pd

## 1.2 Dane kliniczne jako dane tabelaryczne

Biblioteka pandas, którą właśnie zaimportowaliśmy, jest standardowym narzędziem Pythona do pracy z danymi **tabelarycznymi** — a te stanowią bardzo dużą część danych wykorzystywanych w medycynie i biologii. Jest to m. in. naturalna forma zapisu obserwacji klinicznych, w której każdy pacjent opisywany jest przez zestaw mierzalnych cech. W tego typu danych:

* **wiersze** odpowiadają pojedynczym obserwacjom (pacjentom),
* **kolumny** odpowiadają cechom (zmiennym) opisującym te obserwacje.

Podstawowym celem analizy danych tabelarycznych jest:

* zrozumienie struktury danych oraz ich podstawowych właściwości,
* identyfikacja zależności i wzorców pomiędzy zmiennymi,
* porównanie grup pacjentów pod względem wybranych cech.

## 1.3 Przykład: czynniki ryzyka kamicy żółciowej

Kamica żółciowa (obecność kamieni w pęcherzyku żółciowym) jest częstą chorobą przewodu pokarmowego. Jej etiologia jest wieloczynnikowa — na ryzyko zachorowania wpływają czynniki metaboliczne, zapalne, dietetyczne oraz genetyczne. Zrozumienie, które zmienne kliniczne są związane z występowaniem kamicy, może pomóc w identyfikacji pacjentów zagrożonych oraz w planowaniu działań profilaktycznych.

W ramach wykładu przeanalizujemy dane kliniczne zebrane w szpitalu w Ankarze (2022–2023), obejmujące 319 pacjentów — część z rozpoznaną kamicą żółciową, część stanowiącą grupę kontrolną. Dane pochodzą z pracy [Esen et al. (2024)](https://journals.lww.com/md-journal/fulltext/2024/02230/early_prediction_of_gallstone_disease_with_a.40.aspx) i są publicznie dostępne w repozytorium [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/1150/gallstone-1). Zawierają pomiary z trzech kategorii:

* **dane demograficzne** — wiek, płeć, wzrost, waga, BMI,
* **pomiary bioimpedancji** — skład ciała (woda, mięśnie, tkanka tłuszczowa),
* **parametry laboratoryjne** — glukoza, cholesterol, CRP, witamina D i inne.

Naszym celem jest odpowiedź na pytanie: **czym różnią się pacjenci z kamicą żółciową od pacjentów zdrowych?**

### 1.3.1 Wczytanie danych

Dane w formacie CSV (wartości rozdzielone przecinkami) wczytujemy funkcją `read_csv()`. Parametrem tej funkcji jest łańcuch znaków określający ścieżkę do pliku z danymi:

In [2]:
df = pd.read_csv('data/gallstone.csv')

Wynikiem jest obiekt typu `DataFrame` — podstawowa struktura danych w bibliotece pandas, odpowiadająca tabeli. Zmienna `df` jest konwencjonalną nazwą dla ramki danych (od *data frame*), ale często stosuje się również inne nazwy.

### 1.3.2 Pierwszy rzut oka na dane

Zanim przystąpimy do analizy, musimy poznać strukturę danych. Służą do tego podstawowe metody inspekcji.

**Wymiary tabeli:**

In [3]:
df.shape

(319, 39)

Wynik to krotka `(liczba_wierszy, liczba_kolumn)`. Pozwala szybko ocenić rozmiar zbioru danych.

**Podgląd pierwszych wierszy:**

In [4]:
df.head()

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
0,0,50,0,0,0,0,0,0,185,92.8,...,40.0,134.0,20.0,22.0,87.0,0.82,112.47,0.0,16.0,33.0
1,0,47,0,1,0,0,0,0,176,94.5,...,43.0,103.0,14.0,13.0,46.0,0.87,107.10,0.0,14.4,25.0
2,0,61,0,0,0,0,0,0,171,91.1,...,43.0,69.0,18.0,14.0,66.0,1.25,65.51,0.0,16.2,30.2
3,0,41,0,0,0,0,0,0,168,67.7,...,59.0,53.0,20.0,12.0,34.0,1.02,94.10,0.0,15.4,35.4
4,0,42,0,0,0,0,0,0,178,89.6,...,30.0,326.0,27.0,54.0,71.0,0.82,112.47,0.0,16.8,40.6


Metoda `head()` wyświetla domyślnie 5 pierwszych wierszy. Można podać argument, np. `head(10)`, aby zobaczyć więcej. Analogicznie `tail()` pokazuje ostatnie wiersze.

**Informacje o kolumnach:**

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319 entries, 0 to 318
Data columns (total 39 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Gallstone Status                                319 non-null    int64  
 1   Age                                             319 non-null    int64  
 2   Gender                                          319 non-null    int64  
 3   Comorbidity                                     319 non-null    int64  
 4   Coronary Artery Disease (CAD)                   319 non-null    int64  
 5   Hypothyroidism                                  319 non-null    int64  
 6   Hyperlipidemia                                  319 non-null    int64  
 7   Diabetes Mellitus (DM)                          319 non-null    int64  
 8   Height                                          319 non-null    int64  
 9   Weight                                     

Metoda `info()` wyświetla listę wszystkich kolumn wraz z:

* liczbą niepustych wartości (pozwala wykryć braki danych),
* typem danych każdej kolumny (liczby całkowite, zmiennoprzecinkowe, tekst).

**Statystyki opisowe:**

In [6]:
df.describe()

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
count,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000,319.00000,319.000000,...,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000,319.000000
mean,0.495298,48.068966,0.492163,0.335423,0.037618,0.028213,0.025078,0.134796,167.15674,80.564890,...,49.475549,144.502163,21.684953,26.855799,73.112539,0.800611,100.818903,1.853856,14.418182,21.401411
std,0.500763,12.114558,0.500724,0.517340,0.190568,0.165841,0.156609,0.342042,10.05303,15.709069,...,17.718701,97.904493,16.697605,27.884413,24.181069,0.176433,16.971396,4.989591,1.775815,9.981659
min,0.000000,20.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,145.00000,42.900000,...,25.000000,1.390000,8.000000,3.000000,7.000000,0.460000,10.600000,0.000000,8.500000,3.500000
25%,0.000000,38.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,159.50000,69.600000,...,40.000000,83.000000,15.000000,14.250000,58.000000,0.650000,94.170000,0.000000,13.300000,13.250000
50%,0.000000,49.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,168.00000,78.800000,...,46.500000,119.000000,18.000000,19.000000,71.000000,0.790000,104.000000,0.215000,14.400000,22.000000
75%,1.000000,56.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,175.00000,91.250000,...,56.000000,172.000000,23.000000,30.000000,86.000000,0.920000,110.745000,1.615000,15.700000,28.060000
max,1.000000,96.000000,1.000000,3.000000,1.000000,1.000000,1.000000,1.000000,191.00000,143.500000,...,273.000000,838.000000,195.000000,372.000000,197.000000,1.460000,132.000000,43.400000,18.800000,53.100000


Metoda `describe()` oblicza podstawowe statystyki dla kolumn numerycznych: liczność, średnią, odchylenie standardowe, minimum, maksimum oraz kwartyle. Pozwala to szybko ocenić zakresy wartości i wykryć ewentualne anomalie.

Dla lepszej czytelności przy dużej liczbie kolumn można transponować wynik:

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Gallstone Status,319.0,0.495298,0.500763,0.00,0.000,0.000000,1.000,1.00
Age,319.0,48.068966,12.114558,20.00,38.500,49.000000,56.000,96.00
Gender,319.0,0.492163,0.500724,0.00,0.000,0.000000,1.000,1.00
Comorbidity,319.0,0.335423,0.517340,0.00,0.000,0.000000,1.000,3.00
Coronary Artery Disease (CAD),319.0,0.037618,0.190568,0.00,0.000,0.000000,0.000,1.00
Hypothyroidism,319.0,0.028213,0.165841,0.00,0.000,0.000000,0.000,1.00
Hyperlipidemia,319.0,0.025078,0.156609,0.00,0.000,0.000000,0.000,1.00
Diabetes Mellitus (DM),319.0,0.134796,0.342042,0.00,0.000,0.000000,0.000,1.00
Height,319.0,167.156740,10.053030,145.00,159.500,168.000000,175.000,191.00
Weight,319.0,80.564890,15.709069,42.90,69.600,78.800000,91.250,143.50


**Lista nazw kolumn:**

In [8]:
df.columns

Index(['Gallstone Status', 'Age', 'Gender', 'Comorbidity',
       'Coronary Artery Disease (CAD)', 'Hypothyroidism', 'Hyperlipidemia',
       'Diabetes Mellitus (DM)', 'Height', 'Weight', 'Body Mass Index (BMI)',
       'Total Body Water (TBW)', 'Extracellular Water (ECW)',
       'Intracellular Water (ICW)',
       'Extracellular Fluid/Total Body Water (ECF/TBW)',
       'Total Body Fat Ratio (TBFR) (%)', 'Lean Mass (LM) (%)',
       'Body Protein Content (Protein) (%)', 'Visceral Fat Rating (VFR)',
       'Bone Mass (BM)', 'Muscle Mass (MM)', 'Obesity (%)',
       'Total Fat Content (TFC)', 'Visceral Fat Area (VFA)',
       'Visceral Muscle Area (VMA) (Kg)', 'Hepatic Fat Accumulation (HFA)',
       'Glucose', 'Total Cholesterol (TC)', 'Low Density Lipoprotein (LDL)',
       'High Density Lipoprotein (HDL)', 'Triglyceride',
       'Aspartat Aminotransferaz (AST)', 'Alanin Aminotransferaz (ALT)',
       'Alkaline Phosphatase (ALP)', 'Creatinine',
       'Glomerular Filtration Rate (GFR

Zwraca indeks zawierający nazwy wszystkich kolumn. Przydatne, gdy tabela ma wiele zmiennych i chcemy sprawdzić dokładną pisownię nazw.

### 1.3.3 Zmienna docelowa

W naszym zbiorze kolumna `Gallstone Status` oznacza, czy pacjent ma kamicę żółciową (1) czy nie (0). Zanim rozpoczniemy porównania, sprawdźmy rozkład tej zmiennej:

In [9]:
df['Gallstone Status'].value_counts()

Gallstone Status
0    161
1    158
Name: count, dtype: int64

Metoda `value_counts()` zlicza wystąpienia każdej unikalnej wartości. Wynik pokazuje, ile pacjentów należy do każdej grupy.

Aby uzyskać proporcje zamiast liczebności:

In [10]:
df['Gallstone Status'].value_counts(normalize=True)

Gallstone Status
0    0.504702
1    0.495298
Name: proportion, dtype: float64

Zapis `df['Gallstone Status']` wybiera  kolumnę o nazwie `'Gallstone Status'` z ramki danych. Wynikiem jest obiekt typu `Series` — jednowymiarowa struktura danych z indeksem.

## 1.4 Selekcja danych

Analiza danych wymaga umiejętności wybierania interesujących nas fragmentów tabeli — konkretnych kolumn, wierszy spełniających określone warunki, lub ich kombinacji.

### 1.4.1 Wybór kolumn

**Jedna kolumna:**

In [11]:
df['Age']

0      50
1      47
2      61
3      41
4      42
       ..
314    49
315    31
316    58
317    37
318    60
Name: Age, Length: 319, dtype: int64

Wynikiem, jak wcześniej, jest `Series` — wektor wartości z zachowanym indeksem wierszy.

**Kilka kolumn:**

In [12]:
df[['Age', 'Body Mass Index (BMI)', 'Glucose']]

,Age,Body Mass Index (BMI),Glucose
0,50,27.1,102.0
1,47,30.5,94.0
2,61,31.2,103.0
3,41,24.0,69.0
4,42,28.3,109.0
...,...,...,...
314,49,28.0,129.0
315,31,21.7,96.0
316,58,32.7,122.0
317,37,28.2,96.0


Podwójne nawiasy kwadratowe `[[...]]` oznaczają, że przekazujemy listę nazw kolumn. Wynikiem jest `DataFrame` zawierający tylko wybrane kolumny.

### 1.4.2 Wybór wierszy według warunku

Często chcemy wybrać tylko pacjentów spełniających określone kryterium. Służy do tego **indeksowanie logiczne**:

In [13]:
df[df['Gallstone Status'] == 1]

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
161,1,51,0,1,0,0,0,1,165,68.2,...,32.0,91.0,14.0,14.0,86.0,0.73,89.001333,1.25,13.9,35.4
162,1,62,1,1,0,0,0,1,163,83.2,...,44.0,296.0,16.0,18.0,68.0,0.71,91.232000,4.90,12.9,7.9
163,1,30,1,0,0,0,0,0,160,62.3,...,58.0,52.0,16.0,17.0,56.0,0.64,93.462667,0.00,14.0,15.7
164,1,31,1,0,0,0,0,0,160,48.9,...,68.0,23.0,16.0,13.0,54.0,0.59,95.693333,0.50,12.4,3.5
165,1,40,0,1,0,0,0,1,182,110.6,...,42.0,145.0,17.0,36.0,49.0,0.86,97.924000,3.70,15.7,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,1,49,0,0,0,0,0,0,178,88.8,...,55.0,81.0,28.0,50.0,90.0,0.98,94.500000,6.20,16.5,8.3
315,1,31,1,0,0,0,0,0,157,53.4,...,58.0,64.0,24.0,16.0,38.0,0.50,128.500000,0.00,12.5,24.0
316,1,58,0,0,0,0,0,0,172,96.6,...,45.0,168.0,21.0,27.0,94.0,1.04,83.230000,0.00,15.4,15.7
317,1,37,1,0,0,0,0,0,177,88.4,...,33.0,253.0,40.0,22.0,115.0,1.01,98.230000,0.40,16.0,33.3


Wyrażenie `df['Gallstone Status'] == 1` tworzy serię wartości logicznych (`True`/`False`) dla każdego wiersza. Użycie tej serii jako indeksu wybiera tylko wiersze, dla których warunek jest prawdziwy.

**Łączenie warunków:**

In [14]:
df[(df['Gallstone Status'] == 1) & (df['Gender'] == 1)]

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
162,1,62,1,1,0,0,0,1,163,83.2,...,44.0,296.0,16.0,18.0,68.0,0.71,91.232000,4.90,12.9,7.9
163,1,30,1,0,0,0,0,0,160,62.3,...,58.0,52.0,16.0,17.0,56.0,0.64,93.462667,0.00,14.0,15.7
164,1,31,1,0,0,0,0,0,160,48.9,...,68.0,23.0,16.0,13.0,54.0,0.59,95.693333,0.50,12.4,3.5
166,1,76,1,1,0,0,0,0,158,56.4,...,77.0,80.0,29.0,19.0,49.0,0.57,100.154667,3.30,13.5,25.0
167,1,47,1,0,0,0,0,0,175,96.4,...,62.0,50.0,24.0,37.0,63.0,0.54,102.385333,5.20,14.3,27.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,1,47,1,0,0,0,0,0,158,63.6,...,47.0,67.0,16.0,17.0,75.0,0.59,118.200000,0.15,13.9,15.7
313,1,52,1,0,0,0,0,0,150,74.7,...,58.0,68.0,25.0,18.0,66.0,0.75,95.700000,0.00,11.6,12.1
315,1,31,1,0,0,0,0,0,157,53.4,...,58.0,64.0,24.0,16.0,38.0,0.50,128.500000,0.00,12.5,24.0
317,1,37,1,0,0,0,0,0,177,88.4,...,33.0,253.0,40.0,22.0,115.0,1.01,98.230000,0.40,16.0,33.3


Operator `&` oznacza koniunkcję (AND), operator `|` oznacza alternatywę (OR). Każdy warunek musi być ujęty w nawiasy ze względu na priorytety operatorów w Pythonie.

**Przykłady praktyczne:**

Pacjenci z kamicą powyżej 60. roku życia:

In [15]:
df[(df['Gallstone Status'] == 1) & (df['Age'] > 60)]

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
162,1,62,1,1,0,0,0,1,163,83.2,...,44.0,296.0,16.0,18.0,68.0,0.71,91.232000,4.90,12.9,7.9
166,1,76,1,1,0,0,0,0,158,56.4,...,77.0,80.0,29.0,19.0,49.0,0.57,100.154667,3.30,13.5,25.0
184,1,69,1,1,0,0,0,0,158,72.0,...,82.0,181.0,18.0,17.0,78.0,1.09,54.990000,10.70,13.5,23.1
200,1,73,0,1,0,0,0,1,160,104.6,...,47.0,208.0,39.0,64.0,81.0,0.56,83.400000,0.00,14.9,23.8
209,1,64,1,0,0,0,0,0,150,102.1,...,45.0,157.0,13.0,9.0,90.0,0.69,96.800000,2.70,12.0,9.8
213,1,61,1,1,0,0,0,1,152,62.9,...,58.0,117.0,12.0,16.0,83.0,0.70,98.300000,0.80,12.0,21.8
221,1,67,1,1,0,0,0,1,154,70.1,...,47.0,114.0,14.0,18.0,59.0,0.66,96.000000,0.00,12.4,30.0
232,1,66,1,0,0,0,0,0,150,66.4,...,32.0,99.0,16.0,14.0,93.0,0.68,95.990000,6.10,12.9,22.0
234,1,66,0,1,0,0,0,1,170,70.6,...,67.0,71.0,18.0,19.0,70.0,0.94,89.400000,1.10,14.2,9.6
236,1,61,1,0,0,0,0,0,165,60.0,...,52.0,125.0,21.0,18.0,71.0,0.98,65.600000,3.60,14.6,15.2


Pacjenci z **podwyższonym CRP** (> 5 mg/L; próg roboczy do eksploracji) lub z **niską witaminą D** (< 20 ng/mL; próg roboczy):

In [16]:
df[(df['C-Reactive Protein (CRP)'] > 5) | (df['Vitamin D'] < 20)]

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
9,0,38,0,0,0,0,0,0,171,68.6,...,43.0,129.0,19.0,34.0,75.0,0.91,110.63,0.00,16.6,15.60
10,0,34,0,0,0,0,0,0,174,100.8,...,38.0,165.0,23.0,44.0,63.0,0.80,119.10,0.00,15.2,19.00
12,0,23,0,1,0,0,0,0,178,74.1,...,51.0,87.0,22.0,15.0,83.0,1.00,108.46,0.00,18.6,18.80
19,0,35,0,0,0,0,0,0,176,69.3,...,50.0,63.0,21.0,9.0,51.0,1.02,98.29,0.00,16.3,19.16
20,0,50,0,1,0,0,0,1,178,110.7,...,33.0,692.0,24.0,24.0,97.0,0.83,107.28,0.10,17.9,11.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,1,47,1,0,0,0,0,0,158,63.6,...,47.0,67.0,16.0,17.0,75.0,0.59,118.20,0.15,13.9,15.70
312,1,51,0,1,0,0,0,1,173,83.8,...,37.0,212.0,17.0,36.0,128.0,1.29,67.13,0.30,17.0,13.60
313,1,52,1,0,0,0,0,0,150,74.7,...,58.0,68.0,25.0,18.0,66.0,0.75,95.70,0.00,11.6,12.10
314,1,49,0,0,0,0,0,0,178,88.8,...,55.0,81.0,28.0,50.0,90.0,0.98,94.50,6.20,16.5,8.30


### 1.4.3 Metody `loc` i `iloc`

Pandas oferuje dwa dodatkowe sposoby selekcji, które dają większą kontrolę:

**`loc`** — selekcja po **etykietach** (nazwach):

In [17]:
df.loc[0:5, ['Age', 'Body Mass Index (BMI)', 'Gallstone Status']]

,Age,Body Mass Index (BMI),Gallstone Status
0,50,27.1,0
1,47,30.5,0
2,61,31.2,0
3,41,24.0,0
4,42,28.3,0
5,96,20.4,0


Pierwszy argument to zakres wierszy (włącznie z końcem), drugi to lista kolumn.

**`iloc`** — selekcja po **pozycjach** (numerach):

In [18]:
df.iloc[0:5, [0, 1, 2]]

,Gallstone Status,Age,Gender
0,0,50,0
1,0,47,0
2,0,61,0
3,0,41,0
4,0,42,0


Tutaj `0:5` oznacza wiersze od 0 do 4 (bez 5), a `[0, 1, 2]` to numery kolumn.

Różnica jest istotna: `loc` używa nazw i zakresów z włączeniem prawego końca, `iloc` używa pozycji liczbowych i zakresów z wyłączeniem prawego końca (jak standardowe wycinki w Pythonie).

### 1.4.4 Weryfikacja jakości danych

Zanim przejdziemy do analizy porównawczej, warto sprawdzić, czy dane nie zawierają błędnych wpisów. Metoda `describe()` (sekcja 1.3.2) ujawniła podejrzaną wartość: kolumna Obesity (%) ma odchylenie standardowe **109,8** przy medianie **25,6** i maksimum **1954**. Stopień otyłości w procentach nie powinien przyjmować takich wartości.

Odfiltrujmy rekordy z Obesity powyżej 100% i porównajmy je z innymi wskaźnikami otyłości:

In [19]:
df[df['Obesity (%)'] > 100][['Obesity (%)', 'Body Mass Index (BMI)',
                             'Weight', 'Total Body Fat Ratio (TBFR) (%)']]

,Obesity (%),Body Mass Index (BMI),Weight,Total Body Fat Ratio (TBFR) (%)
23,125.6,49.7,143.5,42.30
209,106.3,45.4,102.1,46.70
239,1954.0,29.3,65.9,32.63


Wiersze 23 i 209 dotyczą pacjentek z BMI ~45–50 (otyłość olbrzymia) – wysokie wartości Obesity mogą wynikać ze sposobu pomiaru urządzenia Tanita. Natomiast wiersz 239 to ewidentny błąd: Obesity = 1954 przy BMI zaledwie 29,3 i wadze 65,9 kg. Żaden inny parametr nie wskazuje na ekstremalną otyłość – wartość 1954 została najprawdopodobniej błędnie wpisana (np. rok zamiast pomiaru).

#### Systematyczne wykrywanie błędów

**Wykrycie wszystkich błędów i niespójności wymaga systematycznego przeszukiwania datasetu oraz wiedzy domenowej na temat zamieszczonych cech i jednostek.** To, co zostało znalezione do tej pory, "rzuciło się w oczy" dzięki ekstremalnym wartościom w `describe()`, ale mogą być inne, mniej oczywiste błędy.

Błędy można wykrywać, porównując zmienne, które powinny być ze sobą spójne. Na przykład, całkowita woda w organizmie (TBW) powinna być sumą wody zewnątrzkomórkowej (ECW) i wewnątrzkomórkowej (ICW). Sprawdźmy, czy ta zależność jest spełniona:

In [20]:
# Oblicz różnicę między TBW a sumą ECW+ICW
water_diff = (df["Extracellular Water (ECW)"] + 
              df["Intracellular Water (ICW)"] - 
              df["Total Body Water (TBW)"]).abs()

# Posortuj malejąco i pokaż 10 największych rozbieżności
water_diff.sort_values(ascending=False).head(10)

178    62.6
168    14.7
300    10.0
60      1.0
283     0.5
268     0.5
259     0.4
314     0.4
313     0.4
253     0.4
dtype: float64

To ujawnia dodatkowe niespójności: w wierszach 178, 168 i 300 różnica wynosi odpowiednio 62,6, 14,7 i 10,0 **kg** (≈ litry, jeśli przyjąć 1 kg ≈ 1 L) – co jest znaczącą rozbieżnością. 

Przyjrzyjmy się kilku rekordom z wewnętrznymi niespójnościami:

In [21]:
df.loc[[178], ['Total Body Water (TBW)', 'Extracellular Water (ECW)',
               'Intracellular Water (ICW)', 'Weight']]

,Total Body Water (TBW),Extracellular Water (ECW),Intracellular Water (ICW),Weight
178,13.0,18.5,57.1,79.8


TBW powinno równać się sumie ECW + ICW (18,5 + 57,1 = 75,6 kg), a dla osoby ważącej 79,8 kg oczekiwana wartość TBW to ok. 40–48 kg. Zapisane 13,0 kg jest fizjologicznie niemożliwe – wartości w tym wierszu zostały prawdopodobnie zamienione lub błędnie wpisane.

In [22]:
df.loc[[205], ['Muscle Mass (MM)', 'Weight', 'Lean Mass (LM) (%)']]

,Muscle Mass (MM),Weight,Lean Mass (LM) (%)
205,4.7,78.4,63.14


Lean Mass = 63% z 78,4 kg to ok. 49,5 kg beztłuszczowej masy ciała – wartość 4,7 kg masy mięśniowej jest niemożliwa (prawdopodobnie brakujący przecinek: 47,0 kg).

In [23]:
df.loc[[127, 318], ['Total Cholesterol (TC)', 'High Density Lipoprotein (HDL)',
                     'Low Density Lipoprotein (LDL)', 'Triglyceride']]

,Total Cholesterol (TC),High Density Lipoprotein (HDL),Low Density Lipoprotein (LDL),Triglyceride
127,212.0,56.0,122.0,1.39
318,296.0,273.0,155.0,19.00


Profil lipidowy podlega zależności przybliżonej [wzorem Friedewalda](https://pl.wikipedia.org/wiki/Wz%C3%B3r_Friedewalda): LDL ≈ TC − HDL − TG/5. W wierszu 127 trójglicerydy = 1,39 mg/dL (fizjologicznie niemożliwe; prawdopodobnie 139 mg/dL z błędnym przecinkiem). W wierszu 318 HDL = 273 mg/dL – wartość ponad pięciokrotnie przekraczająca normę; cały profil lipidowy jest wewnętrznie niespójny.

**Usunięcie błędnych rekordów:**

Ponieważ nie znamy prawdziwych wartości, najbezpieczniej jest usunąć te wiersze. Metoda `drop()` tworzy nową ramkę danych bez wskazanych indeksów:

In [24]:
errors = [127, 178, 205, 239, 318]
df = df.drop(errors)
df.shape

(314, 39)

Usunęliśmy 5 z 319 rekordów (1,6%). To niewielka strata, ale jej wpływ na statystyki jest istotny – pojedyncze błędne rekordy potrafią mocno zafałszować statystyki opisowe (np. odchylenie standardowe), dlatego warto je identyfikować przed porównywaniem grup.

> **Uwaga:** w praktyce badawczej usunięcie rekordów wymaga uzasadnienia i udokumentowania. Tutaj usuwamy wyłącznie wiersze z *wewnętrznymi niespójnościami* – wartościami, które są sprzeczne z innymi pomiarami tego samego pacjenta. Wartości skrajnych, ale wewnętrznie spójnych (np. Obesity = 125% przy BMI = 49,7) nie usuwamy.

## 1.5 Grupowanie i agregacja

Kluczową operacją w analizie danych jest **porównywanie grup**. W naszym przypadku chcemy porównać pacjentów z kamicą żółciową i bez niej.

### 1.5.1 Podstawowe grupowanie

In [25]:
df.groupby('Gallstone Status').mean()

,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,Body Mass Index (BMI),...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
Gallstone Status,,,,,,,,,,,,,,,,,,,,,
0,47.712500,0.412500,0.362500,0.056250,0.037500,0.000000,0.100000,168.237500,79.866250,28.256875,...,46.637500,150.350000,23.925000,28.100000,70.837500,0.823781,101.336692,0.465062,14.768125,24.892000
1,48.558442,0.564935,0.311688,0.019481,0.019481,0.051948,0.168831,166.279221,81.580519,29.543506,...,50.946104,140.303896,19.353896,25.253247,75.817532,0.777143,100.394411,3.330325,14.038961,17.727597


Metoda `groupby()` dzieli dane na grupy według wartości wskazanej kolumny. Następnie `mean()` oblicza średnią dla każdej kolumny numerycznej w ramach każdej grupy.

Wynikiem jest ramka danych, gdzie:

* wiersze odpowiadają grupom (0 — brak kamicy, 1 — kamica),
* kolumny to średnie wartości poszczególnych zmiennych.

**Grupowanie z wyborem kolumny:**

In [26]:
df.groupby('Gallstone Status')['Age'].mean()

Gallstone Status
0    47.712500
1    48.558442
Name: Age, dtype: float64

Oblicza średni wiek w grupie. Wynikiem jest `Series`.

**Wiele statystyk jednocześnie:**

In [27]:
df.groupby('Gallstone Status')['C-Reactive Protein (CRP)'].agg(['mean', 'std', 'median'])

,mean,std,median
Gallstone Status,,,
0,0.465062,2.489980,0.0
1,3.330325,6.403956,1.3


In [28]:
df.groupby('Gallstone Status').agg({
    'Age': ['mean', 'std', 'median'],
    'Body Mass Index (BMI)': ['mean', 'std'],
    'Glucose': ['mean', 'std']
})

Age                   Body Mass Index (BMI)            \
                       mean        std median                  mean       std   
Gallstone Status                                                                
0                 47.712500  12.971370   49.0             28.256875  5.301983   
1                 48.558442  11.204567   50.0             29.543506  5.328808   

                     Glucose             
                        mean        std  
Gallstone Status                         
0                 109.306250  48.944327  
1                 107.959091  40.543734

Metoda `agg()` pozwala obliczyć wiele funkcji agregujących naraz. Parametrem jest lista nazw funkcji lub słownik mapujący nazwy kolumn na funkcje.

### 1.5.2 Identyfikacja różnic między grupami

Aby znaleźć zmienne, które najbardziej różnicują pacjentów z kamicą od zdrowych, możemy obliczyć różnicę średnich:

In [29]:
means = df.groupby('Gallstone Status').mean()
means

,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,Body Mass Index (BMI),...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
Gallstone Status,,,,,,,,,,,,,,,,,,,,,
0,47.712500,0.412500,0.362500,0.056250,0.037500,0.000000,0.100000,168.237500,79.866250,28.256875,...,46.637500,150.350000,23.925000,28.100000,70.837500,0.823781,101.336692,0.465062,14.768125,24.892000
1,48.558442,0.564935,0.311688,0.019481,0.019481,0.051948,0.168831,166.279221,81.580519,29.543506,...,50.946104,140.303896,19.353896,25.253247,75.817532,0.777143,100.394411,3.330325,14.038961,17.727597


Obiekt `means` to ramka danych z dwoma wierszami (grupy 0 i 1). Możemy wybrać poszczególne wiersze używając `loc`:

In [30]:
means.loc[1]  # średnie dla grupy z kamicą

Age                                                48.558442
Gender                                              0.564935
Comorbidity                                         0.311688
Coronary Artery Disease (CAD)                       0.019481
Hypothyroidism                                      0.019481
Hyperlipidemia                                      0.051948
Diabetes Mellitus (DM)                              0.168831
Height                                            166.279221
Weight                                             81.580519
Body Mass Index (BMI)                              29.543506
Total Body Water (TBW)                             40.009740
Extracellular Water (ECW)                          16.532468
Intracellular Water (ICW)                          23.324675
Extracellular Fluid/Total Body Water (ECF/TBW)     41.593571
Total Body Fat Ratio (TBFR) (%)                    30.181883
Lean Mass (LM) (%)                                 69.727662
Body Protein Content (Pr

In [31]:
means.loc[0]  # średnie dla grupy kontrolnej

Age                                                47.712500
Gender                                              0.412500
Comorbidity                                         0.362500
Coronary Artery Disease (CAD)                       0.056250
Hypothyroidism                                      0.037500
Hyperlipidemia                                      0.000000
Diabetes Mellitus (DM)                              0.100000
Height                                            168.237500
Weight                                             79.866250
Body Mass Index (BMI)                              28.256875
Total Body Water (TBW)                             41.489375
Extracellular Water (ECW)                          17.642500
Intracellular Water (ICW)                          23.836875
Extracellular Fluid/Total Body Water (ECF/TBW)     42.761750
Total Body Fat Ratio (TBFR) (%)                    26.386750
Lean Mass (LM) (%)                                 73.526750
Body Protein Content (Pr

**Obliczenie różnicy:**

In [32]:
diff = means.loc[1] - means.loc[0]
diff

Age                                                0.845942
Gender                                             0.152435
Comorbidity                                       -0.050812
Coronary Artery Disease (CAD)                     -0.036769
Hypothyroidism                                    -0.018019
Hyperlipidemia                                     0.051948
Diabetes Mellitus (DM)                             0.068831
Height                                            -1.958279
Weight                                             1.714269
Body Mass Index (BMI)                              1.286631
Total Body Water (TBW)                            -1.479635
Extracellular Water (ECW)                         -1.110032
Intracellular Water (ICW)                         -0.512200
Extracellular Fluid/Total Body Water (ECF/TBW)    -1.168179
Total Body Fat Ratio (TBFR) (%)                    3.795133
Lean Mass (LM) (%)                                -3.799088
Body Protein Content (Protein) (%)      

**Czy ten ranking ma sens?**

Pozornie Triglyceride (−10,0) i Vitamin D (−7,2) to zmienne o największej różnicy. Ale porównujemy tu wartości wyrażone w zupełnie różnych skalach i jednostkach. Spadek trójglicerydów o 10 mg/dL ze średniej 150 to niecałe 7% — niewielka zmiana. Tymczasem CRP rośnie z 0,47 do 3,33 mg/L — wzrost o zaledwie 2,9 w wartości bezwzględnej, ale o ponad 600% względem grupy kontrolnej.

Naturalnym pomysłem jest wyrażenie każdej różnicy jako procent średniej w grupie kontrolnej. Pominiemy zmienne binarne (płeć, współchorobowość itp.), dla których zmiana procentowa nie ma sensownej interpretacji:

In [33]:
binary = ['Gender', 'Comorbidity', 'Coronary Artery Disease (CAD)',
          'Hypothyroidism', 'Hyperlipidemia', 'Diabetes Mellitus (DM)']

pct_diff = (diff / means.loc[0]) * 100
pct_diff = pct_diff.drop(binary)

pct_diff.reindex(pct_diff.abs().sort_values(ascending=False).index)

C-Reactive Protein (CRP)                          616.102605
Vitamin D                                         -28.781948
Aspartat Aminotransferaz (AST)                    -19.105972
Hepatic Fat Accumulation (HFA)                     16.730328
Total Fat Content (TFC)                            15.223212
Total Body Fat Ratio (TBFR) (%)                    14.382723
Visceral Fat Area (VFA)                            13.274669
Alanin Aminotransferaz (ALT)                      -10.130794
High Density Lipoprotein (HDL)                      9.238497
Bone Mass (BM)                                     -7.478325
Alkaline Phosphatase (ALP)                          7.030221
Triglyceride                                       -6.681812
Extracellular Water (ECW)                          -6.291809
Creatinine                                         -5.661502
Lean Mass (LM) (%)                                 -5.166946
Hemoglobin (HGB)                                   -4.937417
Body Mass Index (BMI)   

Obraz zmienia się radykalnie. Trójglicerydy, które w rankingu bezwzględnym wydawały się najważniejsze, spadają do −6,7%.

Metoda `reindex()` pozwala uporządkować Series według dowolnego indeksu — tutaj użyliśmy indeksu uzyskanego z sortowania wartości bezwzględnych, dzięki czemu ranking pokazuje zmienne o największej różnicy niezależnie od kierunku, a znak nadal informuje, czy wartość jest wyższa (+) czy niższa (−) w grupie z kamicą.

**Ograniczenia zmiany procentowej**

Zmiana procentowa rozwiązuje problem nieporównywalnych skal, ale ma istotne ograniczenie: zakłada, że średnia w grupie kontrolnej jest sensownym, niezerowym punktem odniesienia. Nie zawsze tak jest.

CRP jest tego dobrym przykładem. Białko C-reaktywne jest *markerem ostrej fazy* — organizm produkuje je w odpowiedzi na stan zapalny, a u zdrowej osoby jego fizjologiczny poziom jest bliski zeru. Przypomnijmy statystyki z sekcji 1.5.1: mediana CRP w grupie kontrolnej wynosi **0,00 mg/L** — ponad połowa zdrowych pacjentów ma CRP niemierzalne. Średnia 0,46 mg/L wynika głównie z pojedynczych pacjentów z subklinicznym stanem zapalnym i nie stanowi sensownej bazy odniesienia. Kiedy dzielimy przez wartość bliską zeru, wynik +616% jest artefaktem — mówi nam jedynie tyle, że „coś bliskie zera wzrosło do czegoś wyraźnie niezerowego".

> **Reguła ogólna:** zmiana procentowa jest użyteczną heurystyką dla zmiennych z niezerowym poziomem bazowym (witamina D, trójglicerydy, BMI). Dla zmiennych, których fizjologiczna norma jest bliska zeru, wynik będzie zawyżony lub niezdefiniowany. Uwaga techniczna: gdybyśmy nie usunęli zmiennych binarnych *przed* obliczeniem, kolumna Hyperlipidemia (ze średnią 0,0 w grupie kontrolnej) dałaby dzielenie przez zero i wartość `inf`.

### 1.5.3 Interpretacja wyników

Traktując zmianę procentową jako przybliżoną heurystykę eksploracyjną — i pamiętając o zastrzeżeniu dotyczącym CRP — możemy wskazać zmienne, które najbardziej różnicują pacjentów z kamicą żółciową od grupy kontrolnej:

* **CRP** — w tym zbiorze średnio wyższe u pacjentów z kamicą. Choć +616% jest artefaktem dzielenia przez wartość bliską zeru, kierunek zmian jest czytelny: CRP to marker odpowiedzi zapalnej, więc może współwystępować z kamicą lub z innymi cechami pacjentów w tej grupie.
* **Witamina D** (−28%) — niższa w grupie z kamicą. Biologicznie to wskaźnik statusu witaminy D (25-OH); w analizie eksploracyjnej traktujemy tę różnicę jako obserwację/hipotezę (możliwe czynniki zakłócające to m.in. wiek, BMI i styl życia).
* **Parametry związane z masą/tłuszczem** — TBFR (+14%), VFA (+13%), BMI (+4,6%) — pokazują spójny wzorzec: w grupie z kamicą wartości są wyższe. To pasuje do intuicji klinicznej, że nadmierna masa ciała i tłuszcz trzewny często współwystępują z chorobami metabolicznymi.

Te obserwacje są spójne z tym, jak zwykle interpretuje się te cechy kliniczne: otyłość jest powszechnie uznawanym czynnikiem ryzyka, a CRP i witamina D częściej traktuje się jako markery/powiązania obserwacyjne. W tym wykładzie zostajemy przy EDA — wnioski przyczynowe wymagają testów i modeli (kolejne wykłady).

Jednocześnie pamiętajmy, że **różnica średnich nie dowodzi związku przyczynowego** — obserwujemy korelacje, które mogą wynikać z czynników zakłócających. Zmiana procentowa nie jest też jedyną metodą normalizacji — w wykładzie 3 poznamy *testy statystyczne*, które pozwolą ocenić, czy zaobserwowane różnice są istotne, oraz *miary wielkości efektu*, które pozwolą porównywać siłę różnic w sposób wolny od problemów zerowego mianownika.

## 1.6 Grupowanie po wielu zmiennych

Czasem chcemy zbadać, czy efekt różni się w podgrupach. Na przykład: czy różnice między chorymi a zdrowymi są takie same u kobiet i mężczyzn?

In [34]:
df.groupby(['Gallstone Status', 'Gender']).mean()

Age  Comorbidity  \
Gallstone Status Gender                           
0                0       47.074468     0.382979   
                 1       48.621212     0.333333   
1                0       47.253731     0.313433   
                 1       49.563218     0.310345   

                         Coronary Artery Disease (CAD)  Hypothyroidism  \
Gallstone Status Gender                                                  
0                0                            0.063830        0.000000   
                 1                            0.045455        0.090909   
1                0                            0.044776        0.014925   
                 1                            0.000000        0.022989   

                         Hyperlipidemia  Diabetes Mellitus (DM)      Height  \
Gallstone Status Gender                                                       
0                0             0.000000                0.117021  174.138298   
                 1             0.000000                0.075758  159.833333   
1                0             0.000000                0.223881  174.626866   
                 1             0.091954                0.126437  159.850575   

                            Weight  Body Mass Index (BMI)  \
Gallstone Status Gender                                     
0                0       84.314894              27.809574   
                 1       73.530303              28.893939   
1                0       87.859701              28.822388   
                 1       76.744828              30.098851   

                         Total Body Water (TBW)  ...  \
Gallstone Status Gender                          ...   
0                0                    46.359574  ...   
                 1                    34.553030  ...   
1                0                    46.852239  ...   
                 1                    34.740230  ...   

                         High Density Lipoprotein (HDL)  Triglyceride  \
Gallstone Status Gender                                                 
0                0                            41.436170    174.202128   
                 1                            54.045455    116.378788   
1                0                            45.271642    151.937313   
                 1                            55.316092    131.344828   

                         Aspartat Aminotransferaz (AST)  \
Gallstone Status Gender                                   
0                0                            26.989362   
                 1                            19.560606   
1                0                            23.305970   
                 1                            16.310345   

                         Alanin Aminotransferaz (ALT)  \
Gallstone Status Gender                                 
0                0                          34.978723   
                 1                          18.303030   
1                0                          35.753731   
                 1                          17.166667   

                         Alkaline Phosphatase (ALP)  Creatinine  \
Gallstone Status Gender                                           
0                0                        68.393617    0.929894   
                 1                        74.318182    0.672652   
1                0                        78.535821    0.884030   
                 1                        73.724138    0.694828   

                         Glomerular Filtration Rate (GFR)  \
Gallstone Status Gender                                     
0                0                             100.113943   
                 1                             103.078182   
1                0                             100.520915   
                 1                             100.296989   

                         C-Reactive Protein (CRP)  Hemoglobin (HGB)  Vitamin D  
Gallstone Status Gender                                                         
0                0             

Wynikiem jest ramka danych z **hierarchicznym indeksem** (MultiIndex). Pandas automatycznie tworzy indeks dwupoziomowy, gdzie pierwszy poziom to `Gallstone`, a drugi to `Gender`.

**Wybór konkretnej zmiennej:**

In [35]:
df.groupby(['Gallstone Status', 'Gender'])['Body Mass Index (BMI)'].mean()

Gallstone Status  Gender
0                 0         27.809574
                  1         28.893939
1                 0         28.822388
                  1         30.098851
Name: Body Mass Index (BMI), dtype: float64

## 1.7 Tabele kontyngencji

Dla zmiennych kategorycznych (binarnych) przydatne są tabele krzyżowe pokazujące liczebności w każdej kombinacji kategorii.

**Czy cukrzyca jest związana z kamicą?**

In [36]:
pd.crosstab(df['Diabetes Mellitus (DM)'], df['Gallstone Status'])

Gallstone Status,0,1
Diabetes Mellitus (DM),,
0,144,128
1,16,26


Funkcja `crosstab()` tworzy tabelę kontyngencji. Wiersze odpowiadają wartościom pierwszej zmiennej, kolumny — drugiej.

**Z proporcjami wierszowymi:**

In [37]:
pd.crosstab(df['Diabetes Mellitus (DM)'], df['Gallstone Status'], normalize='index')

Gallstone Status,0,1
Diabetes Mellitus (DM),,
0,0.529412,0.470588
1,0.380952,0.619048


Normalizacja `'index'` sprawia, że wartości w każdym wierszu sumują się do 1. Pozwala to odpowiedzieć na pytanie: jaki odsetek diabetyków ma kamicę w porównaniu z osobami bez cukrzycy?

Analogicznie można sprawdzić inne choroby współistniejące:

In [38]:
pd.crosstab(df['Hyperlipidemia'], df['Gallstone Status'], normalize='index')

Gallstone Status,0,1
Hyperlipidemia,,
0,0.522876,0.477124
1,0.000000,1.000000


Tablicę kontyngencji możemy wyznaczyć również za pomocą połączenia `groupby()`, agregacji `size()` oraz `unstack()`:

In [39]:
df.groupby(['Gallstone Status', 'Diabetes Mellitus (DM)']).size().unstack(0)

Gallstone Status,0,1
Diabetes Mellitus (DM),,
0,144,128
1,16,26


Metoda `size()` liczy liczbę wystąpień w każdej grupie, a `unstack(0)` przekształca wynik tak, aby wartości pierwszej zmiennej stały się kolumnami.

## 1.8 Sortowanie i ekstrakcja rekordów

Czasem chcemy zidentyfikować konkretnych pacjentów — np. tych z najbardziej ekstremalnymi wartościami.

**Sortowanie:**

In [40]:
df.sort_values('C-Reactive Protein (CRP)', ascending=False)

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
214,1,43,0,0,0,0,0,0,180,81.8,...,31.0,291.0,17.0,29.0,78.0,0.92,105.00,43.4,14.8,18.7
272,1,59,1,0,0,0,0,0,147,82.2,...,61.0,106.0,13.0,14.0,54.0,0.67,10.60,36.7,12.1,48.4
231,1,37,1,0,0,0,0,0,163,74.9,...,55.0,58.0,13.0,9.0,89.5,0.91,83.30,36.1,12.7,6.2
153,0,26,1,1,0,1,0,0,155,74.6,...,34.0,78.0,18.0,16.0,74.0,0.60,107.26,31.0,8.5,27.0
270,1,54,0,0,0,0,0,0,167,72.6,...,52.0,118.0,26.0,17.0,74.0,0.81,104.77,24.5,14.1,5.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,1,52,1,0,0,0,0,0,150,74.7,...,58.0,68.0,25.0,18.0,66.0,0.75,95.70,0.0,11.6,12.1
1,0,47,0,1,0,0,0,0,176,94.5,...,43.0,103.0,14.0,13.0,46.0,0.87,107.10,0.0,14.4,25.0
315,1,31,1,0,0,0,0,0,157,53.4,...,58.0,64.0,24.0,16.0,38.0,0.50,128.50,0.0,12.5,24.0
316,1,58,0,0,0,0,0,0,172,96.6,...,45.0,168.0,21.0,27.0,94.0,1.04,83.23,0.0,15.4,15.7


Zwraca całą ramkę danych posortowaną malejąco według wartości CRP.

**Top N:**

In [41]:
df.nlargest(10, 'C-Reactive Protein (CRP)')

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
214,1,43,0,0,0,0,0,0,180,81.8,...,31.0,291.0,17.0,29.0,78.0,0.92,105.00,43.4,14.8,18.7
272,1,59,1,0,0,0,0,0,147,82.2,...,61.0,106.0,13.0,14.0,54.0,0.67,10.60,36.7,12.1,48.4
231,1,37,1,0,0,0,0,0,163,74.9,...,55.0,58.0,13.0,9.0,89.5,0.91,83.30,36.1,12.7,6.2
153,0,26,1,1,0,1,0,0,155,74.6,...,34.0,78.0,18.0,16.0,74.0,0.60,107.26,31.0,8.5,27.0
270,1,54,0,0,0,0,0,0,167,72.6,...,52.0,118.0,26.0,17.0,74.0,0.81,104.77,24.5,14.1,5.9
268,1,35,1,0,0,0,0,0,165,97.7,...,77.0,73.0,11.0,8.0,71.0,0.57,121.40,21.4,12.6,31.5
257,1,53,1,0,0,0,0,0,165,64.6,...,57.0,78.0,18.0,23.0,115.0,0.94,72.50,18.8,15.0,23.7
280,1,53,1,1,0,0,0,1,150,108.4,...,74.0,97.0,13.0,16.0,77.0,0.65,105.20,16.5,10.9,21.5
277,1,73,0,0,0,0,0,0,181,62.8,...,47.0,116.0,12.0,10.0,74.0,1.16,66.50,14.0,13.0,39.1
208,1,31,0,0,0,0,0,0,178,94.8,...,34.0,70.0,36.0,24.0,67.0,0.72,125.00,13.9,15.4,13.6


Zwraca 10 wierszy o najwyższych wartościach CRP. Jest to szybsze niż sortowanie całej tabeli, gdy interesuje nas tylko kilka rekordów.

**Bottom N:**

In [42]:
df.nsmallest(10, 'Vitamin D')

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
164,1,31,1,0,0,0,0,0,160,48.9,...,68.0,23.0,16.0,13.0,54.0,0.59,95.693333,0.5,12.4,3.5
246,1,48,1,0,0,0,0,0,162,125.8,...,65.0,89.0,12.0,16.0,58.0,0.64,108.900000,12.9,12.1,4.7
156,0,58,1,0,0,0,0,0,163,66.9,...,46.0,122.0,15.0,53.0,101.0,0.65,101.990000,0.0,12.6,4.8
279,1,31,1,0,0,0,0,0,168,93.6,...,45.0,192.0,9.0,12.0,55.0,0.64,121.090000,3.3,14.1,4.9
261,1,44,1,0,0,0,0,0,148,42.9,...,69.0,89.0,18.0,20.0,60.0,0.49,119.100000,0.2,9.4,5.0
286,1,50,1,0,0,0,0,0,160,86.2,...,53.0,162.0,16.0,16.0,65.0,0.69,105.600000,1.4,14.6,5.0
110,0,45,1,0,0,0,0,0,163,61.9,...,77.0,88.0,21.0,11.0,58.0,0.58,114.370000,0.0,13.7,5.1
258,1,52,1,0,0,0,0,0,157,109.2,...,93.0,69.0,15.0,14.0,97.0,0.63,106.600000,1.7,13.7,5.1
283,1,35,1,1,0,0,0,0,173,89.5,...,58.0,66.0,16.0,14.0,56.0,0.74,108.100000,0.2,12.8,5.2
242,1,47,1,1,0,0,0,1,173,87.8,...,33.0,838.0,14.0,16.0,64.0,0.90,79.300000,2.8,14.5,5.3


Zwraca 10 wierszy o najniższych wartościach witaminy D.

## 1.9 Podsumowanie

### Odpowiedź na pytanie badawcze

Przeprowadzona analiza identyfikuje zmienne różniące pacjentów z kamicą żółciową od grupy kontrolnej. Najsilniejsze różnice dotyczą markerów stanu zapalnego (CRP), poziomu witaminy D oraz parametrów związanych z otyłością brzuszną. Jako analiza eksploracyjna, wyniki wskazują sensowne cechy kandydackie do dalszego sprawdzenia (zwłaszcza miary związane z masą/tłuszczem); CRP i witaminę D traktujemy tu jako sygnały do weryfikacji w testach/modelach, a nie jako rozstrzygnięcia.

### Czego jeszcze nie wiemy

Przeprowadzona analiza ma charakter eksploracyjny. Aby odpowiedzieć na dalsze pytania, potrzebujemy:

* **wizualizacji** — wykresy pozwolą lepiej zrozumieć rozkłady i zależności,
* **testów statystycznych** — pozwolą ocenić, czy obserwowane różnice są istotne statystycznie,
* **modeli predykcyjnych** — pozwolą przewidywać ryzyko kamicy u nowych pacjentów.

### Poznane operacje pandas

| Kategoria | Operacje |
|-----------|----------|
| Wczytywanie | `pd.read_csv()` |
| Inspekcja | `head()`, `tail()`, `info()`, `describe()`, `shape`, `columns` |
| Selekcja kolumn | `df['col']`, `df[['col1', 'col2']]` |
| Selekcja wierszy | `df[warunek]`, `loc`, `iloc` |
| Zliczanie | `value_counts()`, `pd.crosstab()`, `size()`, `unstack()` |
| Grupowanie | `groupby()`, `mean()`, `std()`, `agg()` |
| Sortowanie | `sort_values()`, `nlargest()`, `nsmallest()` |

## Dodatek A: Jednostki zmiennych

Plik CSV nie zawiera informacji o jednostkach — to typowa sytuacja w publicznie udostępnianych zbiorach danych. Ustalenie jednostek wymaga sięgnięcia do dokumentacji źródłowej, w tym przypadku do artykułu [Esen et al. (2024)](https://journals.lww.com/md-journal/fulltext/2024/02230/early_prediction_of_gallstone_disease_with_a.40.aspx) oraz specyfikacji analizatora składu ciała Tanita MC780.

Poniższa tabela zestawia jednostki dla wszystkich **39 kolumn** w zbiorze danych (**38 cech + zmienna docelowa** `Gallstone Status`).

### Zmienne demograficzne i kliniczne

| Zmienna | Jednostka | Uwagi |
|---|---|---|
| Gallstone Status | 0/1 | 0 = brak kamicy, 1 = kamica |
| Age | lata | zakres: 20–96 |
| Gender | 0/1 | 0 = mężczyzna, 1 = kobieta |
| Comorbidity | liczba (0–3) | liczba współchorobowości; opis w artykule jest binarny, natomiast w danych zmienna przyjmuje 0–3; interpretujemy ją jako licznik współchorobowości (zmienna porządkowa/licznikowa) |
| Coronary Artery Disease (CAD) | 0/1 | |
| Hypothyroidism | 0/1 | |
| Hyperlipidemia | 0/1 | |
| Diabetes Mellitus (DM) | 0/1 | |

### Pomiary antropometryczne

| Zmienna | Jednostka | Uwagi |
|---|---|---|
| Height | cm | wzrost mierzony boso |
| Weight | kg | waga w lekkiej odzieży (~0,1 kg) |
| Body Mass Index (BMI) | kg/m² | |

### Pomiary bioimpedancji (Tanita MC780)

| Zmienna | Jednostka | Uwagi |
|---|---|---|
| Total Body Water (TBW) | kg | potwierdzone: TBW/Weight ≈ 51% |
| Extracellular Water (ECW) | kg | |
| Intracellular Water (ICW) | kg | TBW ≈ ECW + ICW |
| Extracellular Fluid/Total Body Water (ECF/TBW) | % | stosunek ECW/TBW × 100 |
| Total Body Fat Ratio (TBFR) | % | |
| Lean Mass (LM) | % | |
| Body Protein Content (Protein) | % | |
| Visceral Fat Rating (VFR) | — | bezwymiarowy rating Tanita (skala 1–59) |
| Bone Mass (BM) | kg | mineralna masa kostna |
| Muscle Mass (MM) | kg | |
| Obesity | % | stopień otyłości; **uwaga**: wiersz 239 zawiera wartość 1954 (błąd w danych) |
| Total Fat Content (TFC) | kg | |
| Visceral Fat Area (VFA) | ? | w podanym opisie artykułu VFA figuruje jako kg, jednak w danych zakres i nazwa nie pasują do standardowych cm²; bez specyfikacji Tanita traktujemy VFA jako miarę urządzeniową o nieustalonej jednostce |
| Visceral Muscle Area (VMA) | kg | nazwa kolumny zawiera „(Kg)" |
| Hepatic Fat Accumulation (HFA) | stopień 0–4 | stadium stłuszczenia wątroby wykryte w USG |

### Parametry laboratoryjne (badania krwi)

| Zmienna | Jednostka | Uwagi |
|---|---|---|
| Glucose | mg/dL | |
| Total Cholesterol (TC) | mg/dL | |
| Low Density Lipoprotein (LDL) | mg/dL | |
| High Density Lipoprotein (HDL) | mg/dL | |
| Triglyceride | mg/dL | |
| Aspartat Aminotransferaz (AST) | U/L | |
| Alanin Aminotransferaz (ALT) | U/L | |
| Alkaline Phosphatase (ALP) | U/L | |
| Creatinine | mg/dL | |
| Glomerular Filtration Rate (GFR) | mL/min/1,73 m² | w cytowanym opisie pojawia się ml/seconds; wartości w danych odpowiadają standardowemu eGFR w mL/min/1,73 m² — w dalszej analizie przyjmujemy tę jednostkę |
| C-Reactive Protein (CRP) | mg/L | **uwaga**: nie mg/dL — inna skala niż pozostałe parametry |
| Hemoglobin (HGB) | g/dL | |
| Vitamin D | ng/mL | 25-hydroksywitamina D |

### Uwagi o rozbieżnościach ze źródłem

**GFR** — artykuł podaje jednostkę „ml/seconds", co jest prawdopodobnym błędem redakcyjnym. Wartości w zbiorze danych (zakres 10,6–132, średnia ~101) odpowiadają eGFR raportowanemu klinicznie w standardowej jednostce: mL/min/1,73 m². Gdyby to były mL/s, wartość 100 mL/s oznaczałaby filtrację 6 litrów na minutę — absurd fizjologiczny. W analizie traktujemy tę zmienną jako mL/min/1,73 m².

**Visceral Fat Area (VFA)** — artykuł klasyfikuje VFA wśród zmiennych mierzonych w kilogramach, co stoi w sprzeczności z nazwą (*area*). Zmienna jest ciągła (209 unikalnych wartości z precyzją 0,1 — nie jest więc bezwymiarowym score'em jak VFR) i silnie skorelowana z BMI (r = 0,85), masą tłuszczu (r = 0,87) i wagą (r = 0,88), co wskazuje na sensowny pomiar adipozytowy. Jednak zakres wartości (0,9–41) nie odpowiada typowym wynikom VFA w cm² (rzędu 50–300 w klinicznym CT/MRI). Najprawdopodobniej jest to estymata Tanita MC780 w jednostce specyficznej dla urządzenia; bez dokumentacji producenta nie da się tego jednoznacznie rozstrzygnąć. Jednostka w artykule (kg) jest na pewno niepoprawna.

**Comorbidity** — artykuł opisuje jako zmienną binarną (0/1), ale dane przyjmują wartości 0–3. Nie jest to błąd w danych, lecz niespójność opisu — empiryczna dystrybucja wskazuje na liczbę współistniejących chorób. W analizie traktujemy `Comorbidity` jako zmienną porządkową/licznikową.

**CRP vs inne parametry laboratoryjne** — warto zwrócić uwagę, że CRP jest jedynym parametrem laboratoryjnym wyrażonym w mg/**L** (miligramy na litr), podczas gdy pozostałe (glukoza, cholesterol, kreatynina) podawane są w mg/**dL** (miligramy na decylitr). Wynika to z faktu, że stężenia CRP są zwykle dużo niższe niż stężenia glukozy czy cholesterolu — dlatego w praktyce raportuje się je w mg/L. W niskich zakresach często spotyka się też wartości 0 lub „poniżej oznaczalności” (zależnie od testu), co później ma znaczenie przy analizie procentowych różnic.

## Dodatek B: Kodowanie zmiennej `Gallstone Status`

W niniejszym wykładzie przyjmujemy kodowanie **1 = kamica żółciowa, 0 = grupa kontrolna** (brak kamicy). Jest to zgodne z opisem w sekcji *Materials and Methods* oryginalnego artykułu Esen et al. (2024):

> *„Gallstone status presents information on whether a person has gallstones or not. This feature is a binary variable, where 1 represents a person with gallstones; otherwise 0."*

Niestety, w tym samym artykule (zarówno w abstrakcie, jak i w sekcji opisowej) pojawia się niespójność — podano „161 gallstone patients and 158 healthy controls", co nie zgadza się z kodowaniem opisanym w metodyce (1 = kamica → n=158, a nie 161). Najprawdopodobniej zamieniono liczebności grup, choć nie można wykluczyć innego źródła rozbieżności.

Dokumentacja UCI Machine Learning Repository podaje z kolei `Gallstone Status: 0 (Yes), 1 (No)`, co jest niezgodne zarówno z opisem kodowania w artykule, jak i z weryfikacją przedstawioną poniżej. Możliwe, że odwrócone kodowanie powstało właśnie na podstawie błędnych liczebności z abstraktu (161 przypisano do kodu 0, bo ta grupa liczy 161). Interpretacja z UCI została następnie przejęta przez kolejne publikacje (np. [Sarker et al., 2025](https://www.mdpi.com/1424-8220/25/17/5489)).

### Weryfikacja

Prawidłowość kodowania 1 = kamica można jednoznacznie potwierdzić, porównując rozkład płci w danych z informacją podaną w artykule.

Artykuł podaje, że spośród 157 kobiet w zbiorze, 90 miało kamicę żółciową (a spośród 162 mężczyzn — 68). Tabela krzyżowa `Gender × Gallstone Status`:

```
Gallstone Status    0    1
Gender
0 (mężczyźni)      94   68
1 (kobiety)        67   90
```

Grupa o kodzie 1 zawiera dokładnie 90 kobiet i 68 mężczyzn — zgodnie z artykułem. To potwierdza, że **kod 1 oznacza obecność kamicy**.